<a href="https://colab.research.google.com/github/ekonishi8524/my-colab-notebooks/blob/main/mnist_ddpm_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.optim.lr_scheduler import CosineAnnealingLR

In [ ]:
class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.time_mlp = nn.Sequential(
            nn.Linear(1, 128),
            nn.ReLU(),
            nn.Linear(128, 128)
        )

        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)

        self.down1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU()
        )

        self.down2 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.ReLU()
        )

        self.bottleneck = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(256, 128, kernel_size=3, padding=1), nn.ReLU()
        )

        self.up2 = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1), nn.ReLU()
        )

        self.up1 = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU()
        )

        self.out = nn.Conv2d(64, 1, kernel_size=3, padding=1)

    def forward(self, x, t):
        t_embed = self.time_mlp(t.float().view(-1, 1)).view(-1, 128, 1, 1)

        h1 = self.down1(x)
        h2 = self.pool(h1)
        h2 = self.down2(h2)
        h2 = h2 + t_embed

        h3 = self.pool(h2)
        h3 = self.bottleneck(h3)

        h4 = self.up(h3)
        h4 = torch.cat([h4, h2], dim=1)
        h4 = self.up2(h4)

        h5 = self.up(h4)
        h5 = torch.cat([h5, h1], dim=1)
        h5 = self.up1(h5)

        return self.out(h5)

In [ ]:
import numpy as np

class Diffusion:
    def __init__(self, noise_steps=1000, image_size=28, device="cuda"):
        self.noise_steps = noise_steps
        self.image_size = image_size
        self.device = device

        self.beta = self.prepare_cosine_noise_schedule().to(device)
        self.alpha = 1. - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)

    def prepare_cosine_noise_schedule(self):
        s = 0.008
        steps = self.noise_steps + 1

        t = torch.linspace(0, self.noise_steps, steps)

        alphas_hat = torch.cos(((t / self.noise_steps + s) / (1 + s)) * (np.pi / 2)) ** 2
        alphas_hat = alphas_hat / alphas_hat[0]

        betas = 1.0 - (alphas_hat[1:] / alphas_hat[:-1])

        return torch.clip(betas, 0.0001, 0.999)

    def forward_diffusion(self, x0, t):

        sqrt_alpha_hat = torch.sqrt(self.alpha_hat[t])[:, None, None, None]
        sqrt_one_minus_alpha_hat = torch.sqrt(1. - self.alpha_hat[t])[:, None, None, None]
        epsilon = torch.randn_like(x0)

        xt = sqrt_alpha_hat * x0 + sqrt_one_minus_alpha_hat * epsilon
        return xt, epsilon

    def sample(self, model, n_samples):

        model.eval()
        with torch.no_grad():
            x = torch.randn((n_samples, 1, self.image_size, self.image_size)).to(self.device)

            for i in tqdm(reversed(range(self.noise_steps)), total=self.noise_steps,\
                          desc="Sampling", leave=False):

                t = torch.full((n_samples,), i, dtype=torch.long, device=self.device)

                predicted_noise = model(x, t)

                alpha = self.alpha[t][:, None, None, None]
                alpha_hat = self.alpha_hat[t][:, None, None, None]
                beta = self.beta[t][:, None, None, None]

                mean = (1 / torch.sqrt(alpha)) * (x - (beta / torch.sqrt(1 - alpha_hat))\
                                                  * predicted_noise)

                if i > 0:
                    noise = torch.randn_like(x)
                    sigma = torch.sqrt(beta)
                    x = mean + sigma * noise
                else:
                    x = mean

        model.train()
        x = torch.clamp(x, -1, 1)
        return x

In [ ]:
def train_diffusion_model_400_epochs():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device for training: {device}")

    os.makedirs("outputs", exist_ok=True)

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    dataset = datasets.MNIST(root="dataset", train=True, download=True, transform=transform)

    dataloader = DataLoader(
        dataset,
        batch_size=256,
        shuffle=True,
        num_workers=2,
        pin_memory=True if device == "cuda" else False
    )

    model = SimpleUNet().to(device)
    diffusion = Diffusion(device=device)
    optimizer = optim.Adam(model.parameters(), lr=0.0002)
    criterion = nn.MSELoss()

    scheduler = CosineAnnealingLR(optimizer, T_max=400, eta_min=1e-6)

    print("Starting 400 epochs training with Cosine Annealing Scheduler...")
    epoch_loss = []
    num_epochs = 400

    for epoch in range(num_epochs):
        running_loss = []
        model.train()

        current_lr = optimizer.param_groups[0]['lr']

        for images, _ in tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}\
         (LR: {current_lr:.6f})"):
            images = images.to(device)
            t = torch.randint(0, diffusion.noise_steps, (images.shape[0],), device=device)

            noisy_images, actual_noise = diffusion.forward_diffusion(images, t)
            predicted_noise = model(noisy_images, t)
            loss = criterion(predicted_noise, actual_noise)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss.append(loss.item())

        scheduler.step()

        avg_epoch_loss = sum(running_loss) / len(running_loss)
        epoch_loss.append(avg_epoch_loss)
        print(f"Epoch {epoch+1} average loss: {avg_epoch_loss:.4f}, LR: {current_lr:.6f}")

        current_epoch = epoch + 1
        if current_epoch % 50 == 0 or current_epoch == 1:
            print(f"\n[Epoch {current_epoch}] Generating intermediate samples...")

            sampled_images = diffusion.sample(model, n_samples=10)
            sampled_images = (sampled_images + 1.0) / 2.0
            grid = torchvision.utils.make_grid(sampled_images, nrow=5, padding=2)

            plt.figure(figsize=(10, 4))
            plt.imshow(grid.cpu().permute(1, 2, 0).numpy(), cmap='gray')
            plt.axis('off')
            plt.title(f"Generated Samples at Epoch {current_epoch}")
            plt.savefig(f"outputs/epoch_{current_epoch}.png", bbox_inches='tight')
            plt.close()
            print(f"Saved: outputs/epoch_{current_epoch}.png\n")

    print("Training finished.")

    with open("loss_history.txt", "w") as f:
        for loss in epoch_loss:
            f.write(f"{loss}\n")

    return model, diffusion, epoch_loss

In [ ]:
if __name__ == '__main__':

    trained_model, diffusion_process, epoch_losses = train_diffusion_model_400_epochs()

    plt.figure(figsize=(8, 4))
    plt.plot(epoch_losses, label="Train Loss", color="blue")
    plt.xlabel("Epoch")
    plt.ylabel("Loss (MSE)")
    plt.title("Training Loss Curve")
    plt.grid(True)
    plt.legend()
    plt.show()

    print("\nGenerating final images based on the trained model.")

    sampled_images = diffusion_process.sample(trained_model, n_samples=10)
    sampled_images = (sampled_images + 1.0) / 2.0
    grid = torchvision.utils.make_grid(sampled_images, nrow=5, padding=2)
    grid_np = grid.cpu().permute(1, 2, 0).numpy()

    plt.figure(figsize=(10, 4))
    plt.imshow(grid_np, cmap='gray')
    plt.axis('off')
    plt.title("Final Generated MNIST Samples")
    plt.show()

    torch.save(trained_model.state_dict(), "ddpm_mnist_unet.pth")
    print("Weights of the model are preserved as 'ddpm_mnist_unet.pth'.")

In [ ]:
if __name__ == '__main__':
    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = SimpleUNet().to(device)
    diffusion_process = Diffusion(device=device)

    model.load_state_dict(torch.load("ddpm_mnist_unet.pth", map_location=device))
    print("The trained model has been loaded. You can generate images without training.")

    sampled_images = diffusion_process.sample(model, n_samples=10)

    sampled_images = (sampled_images + 1.0) / 2.0
    grid = torchvision.utils.make_grid(sampled_images, nrow=5, padding=2)
    grid_np = grid.cpu().permute(1, 2, 0).numpy()

    plt.figure(figsize=(10, 4))
    plt.imshow(grid_np, cmap='gray')
    plt.axis('off')
    plt.title("Generated MNIST Samples from Loaded Model")
    plt.show()